# Notebook 15b — Minimal C Sensitivity for DC-Adapted-Oxford102

**Purpose:** Answer the IJITCS reviewer concern that the earlier DC-Adapted baseline used a fixed logistic-regression parameter `C=1.0` while CA-PFG employed validation-based model selection.

**Frozen:** Oxford 102 source dataset, ResNet-50 IMAGENET1K_V2 features, 60 original target tasks (PlantDoc/PlantVillage; K=5/10/20; 10 seeds), target support/validation/test indices, support-fitted 64-D PCA, DC covariance transfer (nearest 2 Oxford classes, source covariance ridge `0.21`), 20 synthetics per support, and logistic-regression training procedure.

**Only new factor:** `C_GRID=(0.01,0.1,1.0,10.0,100.0)`. Fit each C once per task with **identical synthetic features** and choose configuration on validation. Two predetermined selectors are evaluated: (1) **F1-only**, and (2) **the same hierarchical F1→NLL→ECE tolerances as CA-PFG**. Test probabilities are obtained only for the selected model(s). No test labels or CA-PFG test scores determine the selected C.

**Expected outputs:** `300` candidate-validation records, `120` selected test records (60 tasks × two selectors), protocol audit, optional paired comparisons with original fixed-C DC and CA-PFG, and one ZIP.

**How to run:** For a fresh Kaggle notebook, attach the SAME target feature/metadata/manifest dataset as NB15, plus cached Oxford `oxford102_resnet50_features.npy` and `oxford102_metadata.csv` if available. This notebook reuses NB15 cells for data loading and DC generation; it does **not** repeat the previous fixed-C 60-task evaluation. If your prior Kaggle kernel still contains `TARGETS`, `BASE_X`, `BASE_META`, and the NB15 function definitions, you may instead paste only the two new Code Cells below into the end of that notebook.

**Scope note:** The method remains *DC-Adapted-Oxford102*, not the original Distribution Calibration benchmark protocol. Oxford source statistics use independent labeled flower-species images; the baseline therefore uses more labeled source data than support-only CA-PFG. Original NB15 did not run an image-content hash overlap audit. We do not claim otherwise.

> **v1.1 correction before the full run:** the author-facing design had pre-specified `C_LR ∈ {0.01, 0.1, 1, 10, 100}`, whereas the earlier smoke notebook accidentally contained `(1, 10, 30, 100)`. This version restores the pre-specified grid before the 60-task analysis. The change is not based on smoke-test performance.

> **v1.2 assert fix:** the full-run assertion now derives the expected candidate count from `len(C_GRID)`. With five C values and 60 tasks, the expected count is 300 candidates and 120 selected test outcomes.


In [ ]:
# Cell 2 — Config and imports
from __future__ import annotations
import os, sys, json, time, hashlib, warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional
from zipfile import ZipFile, ZIP_DEFLATED
from collections import Counter
import numpy as np
import pandas as pd
from scipy.io import loadmat
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, log_loss, accuracy_score, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression

# RUN STEP 1: False -> one smoke task and audit.
# RUN STEP 2: True -> full 60-task ResNet50 evaluation only after smoke audit.
RUN_FULL = False
BACKBONE = 'ResNet-50'  # primary minimal-baseline experiment
TARGET_DATASETS = ('plantdoc','plantvillage')
SHOTS = (5,10,20)
SEEDS = (137,139,149,151,157,163,167,173,179,181)
SMOKE = ('plantdoc',5,137)
PCA_D = 64
BASE_NEIGHBORS = 2
SYNTH_PER_SUPPORT = 20
FIXED_LR_C = 1.0
LR_MAX_ITER = 1500
COV_REL_RIDGE = 0.21  # scale-relative diagonal stabilization, NOT original constant 0.21 entrywise shift
EIGEN_FLOOR = 1e-7
ECE_BINS = 15
EXTRACTION_BATCH_SIZE = 32
EXTRACTION_WORKERS = 2
ALLOW_CPU_EXTRACTION = False

BASE_NAME = 'oxford_flowers102'
BASE_EXPECTED_CLASSES = 102
BASE_EXPECTED_IMAGES = 8189

ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUTPUT_DIR = ROOT/'CAPFG15_DC_Adapted_Output'
OUTPUT_DIR.mkdir(exist_ok=True,parents=True)
BASE_CACHE = ROOT/'capfg15_aux_cache'
BASE_CACHE.mkdir(exist_ok=True,parents=True)
SEARCH_ROOTS = [Path('/kaggle/input'),ROOT,Path('/mnt/data')]

# Manually set paths if auto-discovery finds ambiguous copies.
MANUAL = {
    'plantdoc_features': None,    # plantdoc_resnet50_features.npy
    'plantdoc_metadata': None,    # plantdoc_resnet50_metadata.csv OR plantdoc_metadata.csv
    'plantvillage_features': None,
    'plantvillage_metadata': None,
    'manifest': None,             # manifest_split.csv
    'oxford_features': None,      # oxford102_resnet50_features.npy if previously extracted
    'oxford_metadata': None,      # oxford102_metadata.csv
    'oxford_root': None,          # directory with jpg/ + imagelabels.mat
    'resnet_weights': None,       # optional local TorchVision IMAGENET1K_V2 .pth
    'test_reference': None,       # normalized_all_results_TEST_ONLY.csv from NB12 v3
}
print('Notebook 15, Run full =',RUN_FULL,'output =',OUTPUT_DIR)


In [ ]:
# Cell 3 — Input resolution (never select data from target output as independent base)
def find_input_file(basename, manual_key=None, required=True):
    if manual_key and MANUAL.get(manual_key):
        p=Path(MANUAL[manual_key]); assert p.is_file(),f'Missing {p}'
        return p
    found=[]
    for root in SEARCH_ROOTS:
        if root.exists():
            found.extend(p for p in root.rglob(basename)
                         if OUTPUT_DIR.resolve() not in p.resolve().parents)
    found=sorted(set(found), key=lambda p:(0 if '/kaggle/input/' in str(p) else 1, len(p.parts),str(p)))
    if not found:
        if required: raise FileNotFoundError(f'Cannot find {basename}. Attach the original feature package as Kaggle Input.')
        return None
    if len(found)>1:
        print(f'[INPUT WARN] Multiple {basename}; using {found[0]}')
    return found[0]

def get_meta_path(dataset):
    if MANUAL.get(f'{dataset}_metadata'):
        p=Path(MANUAL[f'{dataset}_metadata']); assert p.exists(); return p
    return (find_input_file(f'{dataset}_resnet50_metadata.csv', required=False)
            or find_input_file(f'{dataset}_metadata.csv'))

def get_feat_path(dataset):
    # Prefer the canonical Notebook 11 ResNet-50 feature cache name.
    # Allow a generic filename ONLY when its metadata/dimension is compatible.
    manual_key = f'{dataset}_features'
    if MANUAL.get(manual_key):
        candidates = [Path(MANUAL[manual_key])]
    else:
        canonical = find_input_file(f'{dataset}_resnet50_features.npy', required=False)
        generic = find_input_file(f'{dataset}_features.npy', required=False)
        candidates = [p for p in (canonical, generic) if p is not None]

    if not candidates:
        raise FileNotFoundError(
            f'{dataset}: missing ResNet-50 feature file. Expected '
            f'{dataset}_resnet50_features.npy (Notebook 11 output); '
            f'also checked {dataset}_features.npy. '
            'Attach the real Notebook 11 cache; DINOv2 384-d features are NOT compatible.'
        )

    diagnostics = []
    for path in candidates:
        if not path.is_file():
            raise FileNotFoundError(f'Manual feature path does not exist: {path}')
        x = np.load(path, mmap_mode='r', allow_pickle=False)
        diagnostics.append(f'{path.name}: shape={x.shape}')
        if x.ndim == 2 and x.shape[1] == 2048:
            print(f'[FEATURE CHECK] {dataset}: {path.name}, shape={x.shape}, '
                  'dim=2048 (ResNet-50 compatible)')
            if path.name == f'{dataset}_features.npy':
                print('[PROVENANCE NOTE] Generic filename accepted by dimension check. '
                      'Please confirm it was produced by Notebook 11 with '
                      'ResNet50 IMAGENET1K_V2; dimensions alone do not prove provenance.')
            return path

    raise ValueError(
        f'{dataset}: no 2048-dimensional ResNet-50 feature cache found. '
        + '; '.join(diagnostics)
        + '. If these are 384-dimensional DINOv2 features from Notebook 2, '
          'DO NOT rename them. Attach the original Notebook 11 ResNet-50 caches.'
    )

def sha256_short(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for buf in iter(lambda:f.read(2**20),b''): h.update(buf)
    return h.hexdigest()

def canonical_dataset(value):
    value=str(value).strip().lower().replace('_','').replace(' ','').replace('-','')
    return 'plantdoc' if 'plantdoc' in value else ('plantvillage' if 'plantvillage' in value else value)

def canonical_split(value):
    v=str(value).strip().lower()
    return {'val':'validation','valid':'validation','dev':'validation','training':'train','tr':'train','te':'test','testing':'test'}.get(v,v)

def choose_col(df,names):
    mapping={str(c).lower():c for c in df.columns}
    return next((mapping[n] for n in names if n in mapping),None)

def canonical_metadata(df, dataset, manifest):
    m=df.copy().reset_index(drop=True)
    # Preserve exact original Notebook 11 row/feature index. Avoid sorting.
    lcol=choose_col(m,['label_canonical','label_idx','class_idx','class_id','target','y','label'])
    scol=choose_col(m,['split_canonical','split','subset','partition'])
    idcol=choose_col(m,['image_id','sample_id','id','uid'])
    pathcol=choose_col(m,['relative_path','rel_path','image_path','filepath','path','filename'])
    if (not lcol or not scol) and manifest is not None:
        mm=manifest.copy()
        dcol=choose_col(mm,['dataset','dataset_name'])
        if dcol: mm=mm[mm[dcol].map(canonical_dataset)==dataset]
        mid=choose_col(mm,['image_id','sample_id','id','uid'])
        mp=choose_col(mm,['relative_path','rel_path','image_path','filepath','path','filename'])
        if idcol and mid:
            key1=m[idcol].astype(str); key2=mm[mid].astype(str)
        elif pathcol and mp:
            key1=m[pathcol].astype(str).str.replace('\\','/',regex=False)
            key2=mm[mp].astype(str).str.replace('\\','/',regex=False)
        else:
            raise ValueError(f'{dataset}: Cannot harmonize missing label/split without matching IDs')
        m['_link']=key1; mm=mm.copy();mm['_link']=key2
        assert not m['_link'].duplicated().any() and not mm['_link'].duplicated().any(), 'Nonunique manifest keys'
        m=m.merge(mm.drop_duplicates('_link'),on='_link',how='left',suffixes=('','_manifest'),sort=False)
        lcol=choose_col(m,['label_canonical','label_idx','class_idx','class_id','target','y','label','label_manifest'])
        scol=choose_col(m,['split_canonical','split','subset','partition','split_manifest'])
    if not lcol or not scol:
        raise ValueError(f'{dataset}: no usable split or label field: {list(m.columns)}')
    numeric=pd.to_numeric(m[lcol],errors='coerce')
    if numeric.notna().all(): m['label_canonical']=numeric.astype(int)
    else: m['label_canonical']=pd.Categorical(m[lcol].astype(str), categories=sorted(m[lcol].astype(str).unique())).codes
    m['split_canonical']=m[scol].map(canonical_split)
    if not set(m['split_canonical'].unique()) <= {'train','validation','test'}:
        raise ValueError(f'{dataset}: Unrecognized splits {m.split_canonical.unique()}')
    m['feature_index']=np.arange(len(m),dtype=int)
    m['sample_key']=m[idcol].astype(str) if idcol else m['feature_index'].astype(str)
    if m['label_canonical'].isna().any(): raise ValueError('Missing labels')
    return m

@dataclass
class TargetBundle:
    dataset: str
    features: np.ndarray
    metadata: pd.DataFrame
    class_labels: np.ndarray

def load_targets():
    manifest_path=find_input_file('manifest_split.csv','manifest',required=False)
    manifest=pd.read_csv(manifest_path) if manifest_path else None
    bundles={}; audits=[]
    for dataset in TARGET_DATASETS:
        fp=get_feat_path(dataset); mp=get_meta_path(dataset)
        X=np.load(fp,mmap_mode='r')
        m=canonical_metadata(pd.read_csv(mp),dataset,manifest)
        assert len(X)==len(m),f'{dataset}: feature/metadata row count mismatch'
        assert X.ndim==2 and X.shape[1]==2048,f'{dataset}: expected ResNet50 2048 features, got {X.shape}'
        assert np.isfinite(X).all(),f'{dataset}: nonfinite target features'
        cls=np.sort(m.label_canonical.unique()); expected={'plantdoc':27,'plantvillage':38}[dataset]
        assert len(cls)==expected,(dataset,len(cls),expected)
        expected_layout={'plantdoc':(8887,7175,1268,444),
                         'plantvillage':(54284,38003,5430,10851)}[dataset]
        actual_layout=(len(X),*(int((m.split_canonical==spl).sum()) for spl in ('train','validation','test')))
        assert actual_layout==expected_layout, f'{dataset}: input split mismatch: {actual_layout} vs {expected_layout}'
        bundles[dataset]=TargetBundle(dataset,np.asarray(X,dtype=np.float32),m,cls)
        audits.append(dict(dataset=dataset,rows=len(X),classes=len(cls),feature_dim=X.shape[1],
                           train_count=int((m.split_canonical=='train').sum()),
                           validation_count=int((m.split_canonical=='validation').sum()),
                           test_count=int((m.split_canonical=='test').sum()),
                           source_file=str(fp),feature_sha256=sha256_short(fp)))
    pd.DataFrame(audits).to_csv(OUTPUT_DIR/'target_input_audit.csv',index=False)
    return bundles

TARGETS=load_targets()
print('Target feature bundles loaded:',{k:v.features.shape for k,v in TARGETS.items()})


## Phase A — Source-only Oxford Flowers 102 data

Attach a Kaggle Input containing `oxford102_resnet50_features.npy` and `oxford102_metadata.csv` **if these caches already exist**. Otherwise the cell below will load the official Oxford images (or download them with Kaggle Internet enabled) and extract ResNet-50 IMAGENET1K_V2 features **once**. It must not use PlantDoc/PlantVillage images to build base-class statistics.

The official Oxford Flowers102 archive contains 8,189 images, 102 species classes. Its full public corpus is used to estimate auxiliary source-class statistics; it is **not** evaluated as a novel/test dataset in this experiment.


In [ ]:
# Cell 5 — Load or download Oxford 102 labels/images; NO target data
from PIL import Image

def oxford_raw_root():
    if MANUAL.get('oxford_root'):
        p=Path(MANUAL['oxford_root'])
        if not (p/'imagelabels.mat').exists() or not (p/'jpg').exists():
            raise ValueError('oxford_root must contain imagelabels.mat and jpg/')
        return p
    for root in (Path('/kaggle/input'), ROOT):
        if root.exists():
            found=list(root.rglob('imagelabels.mat'))
            for f in found:
                if (f.parent/'jpg').is_dir(): return f.parent
    # Download via official TorchVision dataset helper.
    try:
        from torchvision.datasets import Flowers102
        out=BASE_CACHE/'flowers_download'
        print('[DOWNLOAD] Official Oxford Flowers102. Kaggle Internet may be needed.')
        Flowers102(root=str(out),split='train',download=True)
        for p in out.rglob('imagelabels.mat'):
            if (p.parent/'jpg').is_dir(): return p.parent
    except Exception as exc:
        raise RuntimeError('Oxford102 images not found. Attach Oxford data to Kaggle Input or enable Internet. '+str(exc)) from exc
    raise FileNotFoundError('Oxford102 jpg/ and imagelabels.mat missing after attempted download')

def load_oxford_image_metadata():
    base=oxford_raw_root()
    labels=np.asarray(loadmat(base/'imagelabels.mat')['labels']).ravel().astype(int)
    assert len(labels)==BASE_EXPECTED_IMAGES,(len(labels),BASE_EXPECTED_IMAGES)
    paths=[base/'jpg'/f'image_{i:05d}.jpg' for i in range(1,len(labels)+1)]
    missing=[str(p) for p in paths if not p.is_file()]
    if missing: raise FileNotFoundError(f'{len(missing)} Oxford images not found; first: {missing[0]}')
    counts=pd.Series(labels).value_counts()
    assert len(counts)==BASE_EXPECTED_CLASSES and counts.min()>=40 and counts.max()<=258,counts.describe()
    frame=pd.DataFrame({'source_dataset':BASE_NAME,'sample_id':[p.stem for p in paths],
                        'source_class_id':labels,'image_path':[str(p) for p in paths]})
    return paths,frame

def select_device():
    import torch
    if not torch.cuda.is_available():
        if ALLOW_CPU_EXTRACTION: return torch.device('cpu')
        raise RuntimeError('Enable a compatible Kaggle GPU to extract Oxford102 features (or attach cached features)')
    capability=torch.cuda.get_device_capability(0)
    arch=f'sm_{capability[0]}{capability[1]}'
    supported=torch.cuda.get_arch_list()
    print('[GPU]',torch.cuda.get_device_name(0),'arch',arch,'supported',supported)
    if arch not in supported:
        if ALLOW_CPU_EXTRACTION: return torch.device('cpu')
        raise RuntimeError('GPU/PyTorch architecture mismatch. Use T4 or a compatible accelerator; do not run on mismatched P100.')
    return torch.device('cuda')

def extract_oxford_resnet_features(paths):
    import torch
    import torch.nn as nn
    from torchvision.models import resnet50, ResNet50_Weights
    from torch.utils.data import Dataset, DataLoader
    weights=ResNet50_Weights.IMAGENET1K_V2
    dev=select_device()
    model=resnet50(weights=None if MANUAL.get('resnet_weights') else weights)
    if MANUAL.get('resnet_weights'):
        try: ckpt=torch.load(MANUAL['resnet_weights'],map_location='cpu',weights_only=True)
        except TypeError: ckpt=torch.load(MANUAL['resnet_weights'],map_location='cpu')
        model.load_state_dict(ckpt)
    model.fc=nn.Identity();model.eval().to(dev)
    tfm=weights.transforms()  # EXACT same preprocessing as NB11
    class Images(Dataset):
        def __len__(self): return len(paths)
        def __getitem__(self,j):
            with Image.open(paths[j]) as im: return tfm(im.convert('RGB')),j
    loader=DataLoader(Images(),batch_size=EXTRACTION_BATCH_SIZE,shuffle=False,
                      num_workers=EXTRACTION_WORKERS,pin_memory=dev.type=='cuda')
    feats=np.zeros((len(paths),2048),dtype=np.float32); seen=np.zeros(len(paths),bool)
    with torch.inference_mode():
        for bi,(X,ind) in enumerate(loader):
            X=X.to(dev,non_blocking=True)
            pred=model(X).float().cpu().numpy()
            feats[ind.numpy()]=pred;seen[ind.numpy()]=True
            if bi%40==0 or bi==len(loader)-1: print('[EXTRACT]',bi+1,'/',len(loader))
    assert seen.all() and np.isfinite(feats).all()
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return feats

base_feat_file=(Path(MANUAL['oxford_features']) if MANUAL.get('oxford_features')
                else find_input_file('oxford102_resnet50_features.npy',required=False))
base_meta_file=(Path(MANUAL['oxford_metadata']) if MANUAL.get('oxford_metadata')
                else find_input_file('oxford102_metadata.csv',required=False))
if base_feat_file and base_meta_file:
    BASE_X=np.asarray(np.load(base_feat_file),dtype=np.float32)
    BASE_META=pd.read_csv(base_meta_file)
    print('Using Oxford auxiliary cached features:',base_feat_file)
else:
    IMG_PATHS, BASE_META=load_oxford_image_metadata()
    BASE_X=extract_oxford_resnet_features(IMG_PATHS)
    np.save(BASE_CACHE/'oxford102_resnet50_features.npy',BASE_X)
    BASE_META.to_csv(BASE_CACHE/'oxford102_metadata.csv',index=False)
    base_feat_file=BASE_CACHE/'oxford102_resnet50_features.npy'
    base_meta_file=BASE_CACHE/'oxford102_metadata.csv'
assert 'source_class_id' in BASE_META.columns,BASE_META.columns
assert len(BASE_X)==len(BASE_META)==BASE_EXPECTED_IMAGES,(BASE_X.shape,BASE_META.shape)
assert BASE_X.shape[1]==2048 and np.isfinite(BASE_X).all()
source_class_counts=BASE_META.source_class_id.value_counts()
assert len(source_class_counts)==102 and source_class_counts.min()>=40
# Semantic source labels: Oxford flower species; NOT PlantDoc/PlantVillage disease categories.
source_audit={ 'dataset':BASE_NAME,'n_images':int(len(BASE_X)),
              'n_species':int(len(source_class_counts)),'min_per_species':int(source_class_counts.min()),
              'max_per_species':int(source_class_counts.max()),'feature_dim':int(BASE_X.shape[1]),
              'feature_file':str(base_feat_file), 'metadata_file':str(base_meta_file),
              'feature_sha256':sha256_short(base_feat_file),
              'target_image_hash_overlap_checked':False,
              'source_and_target_label_spaces_are_distinct_by_semantics':True,
              'source_base_is_cross_domain':True}
(OUTPUT_DIR/'base_data_provenance.json').write_text(json.dumps(source_audit,indent=2),encoding='utf-8')
print('Oxford 102 source:',source_audit)


## Phase B — Independent task evaluation

All target preprocessing is support-fitted. The independent Oxford source is transformed through the **same task-specific support PCA** before source mean/covariance estimation. Validation is intentionally **not used** by fixed-configuration DC-Adapted; test is evaluated once per task. Baseline hyperparameters are fixed *before* running and not adjusted according to CA-PFG test scores.


In [ ]:
# Cell 7 — Exact Notebook 11 support split and 64-D PCA
import hashlib

def stable_int_hash(value,modulo=2**32-1):
    digest=hashlib.sha256(str(value).encode('utf-8')).hexdigest()
    return int(digest[:16],16)%modulo

def derive_seed(base_seed,component_name):
    return stable_int_hash(f'{int(base_seed)}::{component_name}')

def build_task_split(bundle,k,seed):
    meta=bundle.metadata
    train=meta.loc[meta['split_canonical']=='train']
    valid=meta.index[meta['split_canonical']=='validation'].to_numpy(dtype=int)
    test=meta.index[meta['split_canonical']=='test'].to_numpy(dtype=int)
    parts=[]
    for label in bundle.class_labels:
        candidates=train.index[train['label_canonical']==label].to_numpy(dtype=int)
        if len(candidates)<k: raise ValueError(f'{bundle.dataset} class {label} has insufficient train samples')
        rng=np.random.default_rng(derive_seed(seed,f'support::{bundle.dataset}::{k}::{label}'))
        parts.append(np.sort(rng.choice(candidates,size=k,replace=False)))
    support=np.sort(np.concatenate(parts))
    assert not (set(support)&set(valid)) and not (set(support)&set(test)) and not (set(valid)&set(test))
    y=meta.loc[support,'label_canonical'].to_numpy(dtype=int)
    observed,counts=np.unique(y,return_counts=True)
    assert np.array_equal(observed,bundle.class_labels) and np.all(counts==k)
    return support,valid,test

def support_fitted_pca(raw_support, seed):
    X=normalize(raw_support,norm='l2')
    d=min(PCA_D,len(X)-1,X.shape[1]);assert d>=2
    # Original NB11: whiten=False, svd_solver='full'
    pca=PCA(n_components=d,whiten=False,svd_solver='full',random_state=derive_seed(seed,'pca'))
    pca.fit(X)
    return pca

def convert_with_pca(raw,pca):
    result=pca.transform(normalize(raw,norm='l2'))
    assert np.isfinite(result).all()
    return np.asarray(result,dtype=np.float64)

# Integrity checks: verify task keys, cohort sizes and 5-shot PCA dimensionality.
for dataset,b in TARGETS.items():
    s,v,t=build_task_split(b,5,137)
    print('[TARGET AUDIT]',dataset,'support',len(s),'val',len(v),'test',len(t),
          'n_classes',len(b.class_labels),'PCA cap',min(PCA_D,len(s)-1,2048))


In [ ]:
# Cell 8 — DC-adapted transfer with positive-definite covariance audit
from scipy.linalg import cholesky

def estimate_oxford_source_stats(X_base_64,y_base):
    classes=np.sort(np.unique(y_base))
    means=[];covs=[]
    for cls in classes:
        X=np.asarray(X_base_64[y_base==cls],dtype=np.float64)
        assert len(X)>=40 and X.shape[1]==PCA_D
        mu=X.mean(axis=0);centered=X-mu
        cov=(centered.T@centered)/max(1,len(X)-1)
        means.append(mu);covs.append((cov+cov.T)*0.5)
    return classes,np.asarray(means),np.asarray(covs)

def calibrate_support_dc(z,base_mu,base_cov,k_nearest=BASE_NEIGHBORS,chol_cache=None):
    # Base-class mean-neighbor transfer, following the central DC mechanism.
    dist=np.sum((base_mu-z[None,:])**2,axis=1)
    indices=np.argsort(dist,kind='stable')[:k_nearest]
    calib_mu=(np.sum(base_mu[indices],axis=0)+z)/(k_nearest+1)
    key=tuple(sorted(int(i) for i in indices))
    if chol_cache is not None and key in chol_cache:
        L=chol_cache[key]
    else:
        raw_cov=np.mean(base_cov[indices],axis=0)
        # This scale-aware ridge is a disclosed numerical adaptation to 64-D PCA.
        trace_scale=max(float(np.trace(raw_cov))/len(z),EIGEN_FLOOR)
        calib_cov=(raw_cov+raw_cov.T)*0.5 + max(COV_REL_RIDGE*trace_scale,EIGEN_FLOOR)*np.eye(len(z))
        try:
            L=np.linalg.cholesky(calib_cov)
        except np.linalg.LinAlgError:
            # Rare finite precision issue; not a model search/hyperparameter tune.
            w,v=np.linalg.eigh(calib_cov)
            calib_cov=(v*np.maximum(w,EIGEN_FLOOR))@v.T
            L=np.linalg.cholesky((calib_cov+calib_cov.T)*0.5+EIGEN_FLOOR*np.eye(len(z)))
        if chol_cache is not None:chol_cache[key]=L
    return calib_mu,L,indices

def gen_dc_features(X_s,y_s,base_mu,base_cov,task_seed):
    all_synth=[];all_labels=[];nn_indices=[];chol_cache={}
    for j,(z,y) in enumerate(zip(X_s,y_s)):
        mu,L,near=calibrate_support_dc(z,base_mu,base_cov,chol_cache=chol_cache)
        nn_indices.extend(near.tolist())
        rng=np.random.default_rng(derive_seed(task_seed,f'dc_synth::{j}'))
        gen=mu[None,:] + rng.standard_normal((SYNTH_PER_SUPPORT,len(mu)))@L.T
        all_synth.append(gen);all_labels.extend([int(y)]*SYNTH_PER_SUPPORT)
    return np.vstack(all_synth),np.array(all_labels,dtype=int),nn_indices

def brier(y,p,classes):
    pos={int(c):j for j,c in enumerate(classes)}
    onehot=np.zeros_like(p);onehot[np.arange(len(y)),[pos[int(v)] for v in y]]=1
    return float(np.mean(np.sum((p-onehot)**2,axis=1)))

def ece(y,p,classes,n_bins=15):
    pred=classes[np.argmax(p,axis=1)];conf=np.max(p,axis=1)
    acc=(pred==y).astype(float);edges=np.linspace(0,1,n_bins+1);s=0.0
    for j in range(n_bins):
        mask=((conf>=edges[j]) if j==0 else (conf>edges[j])) & (conf<=edges[j+1])
        if np.any(mask):s+=mask.mean()*abs(acc[mask].mean()-conf[mask].mean())
    return float(s)

def get_metrics(y,p,classes):
    pred=classes[np.argmax(p,axis=1)]
    return dict(macro_f1=float(f1_score(y,pred,labels=classes,average='macro',zero_division=0)),
                accuracy=float(accuracy_score(y,pred)),
                balanced_accuracy=float(balanced_accuracy_score(y,pred)),
                nll=float(log_loss(y,p,labels=classes)),ece=ece(y,p,classes,ECE_BINS),
                brier=brier(y,p,classes))

def evaluate_dc_task(bundle,k,seed,record_predictions=False):
    tick=time.perf_counter()
    support,valid,test=build_task_split(bundle,k,seed)
    m=bundle.metadata;classes=bundle.class_labels
    Xs=np.asarray(bundle.features[support],dtype=np.float64)
    Xt=np.asarray(bundle.features[test],dtype=np.float64)
    ys=m.loc[support,'label_canonical'].to_numpy(dtype=int)
    yt=m.loc[test,'label_canonical'].to_numpy(dtype=int)
    pca=support_fitted_pca(Xs,seed)
    Zs=convert_with_pca(Xs,pca);Zt=convert_with_pca(Xt,pca)
    # Base statistics are derived exclusively from external Oxford102 features.
    Zb=convert_with_pca(BASE_X,pca)
    base_y=BASE_META.source_class_id.to_numpy(dtype=int)
    _,base_mu,base_cov=estimate_oxford_source_stats(Zb,base_y)
    Xgen,ygen,neighbor_indices=gen_dc_features(Zs,ys,base_mu,base_cov,seed)
    assert len(Xgen)==len(ys)*SYNTH_PER_SUPPORT
    clf=LogisticRegression(C=FIXED_LR_C,solver='lbfgs',max_iter=LR_MAX_ITER,
                           random_state=derive_seed(seed,'external_dc_fixed_lr'))
    clf.fit(np.vstack([Zs,Xgen]),np.concatenate([ys,ygen]))
    prob=clf.predict_proba(Zt)
    assert np.array_equal(clf.classes_,classes),'Logistic model classes differ from expected target labels'
    assert np.isfinite(prob).all() and np.allclose(prob.sum(axis=1),1,atol=1e-8)
    met=get_metrics(yt,prob,classes)
    row=dict(backbone=BACKBONE,dataset=bundle.dataset,shot=k,K=k,seed=seed,
             task_key=f'{bundle.dataset}__k{k}__seed{seed}',method='DC-Adapted-Oxford102',
             **{f'test_{key}':v for key,v in met.items()},
             n_support=len(support),n_validation_unused=len(valid),n_test=len(test),
             pca_components=int(pca.n_components_),n_source_classes=len(base_mu),
             n_source_samples=len(BASE_X),n_synthetic=len(Xgen),
             synthetic_multiplier=SYNTH_PER_SUPPORT,
             cv_selection_used=False,chosen_C=FIXED_LR_C,DC_k=BASE_NEIGHBORS,
             nearest_source_class_uses_target_label=False,
             elapsed_sec=round(time.perf_counter()-tick,2))
    audit=dict(task_key=row['task_key'],backbone=BACKBONE,source_dataset=BASE_NAME,
               target_test_directly_used_for_base=False,
               source_target_image_hash_overlap_checked=False,
               base_labels_target_label_used=False,
               target_support_only_for_pca=True,
               target_other_train_used_for_base=False,
               validation_used_for_selection=False,
               target_test_labels_used_for_optimization=False,
               source_sample_count=len(BASE_X),
               n_unique_source_nn_classes=len(set(neighbor_indices)),
               support_sample_indices_sha256=hashlib.sha256(np.asarray(support,dtype=np.int64).tobytes()).hexdigest(),
               test_sample_indices_sha256=hashlib.sha256(np.asarray(test,dtype=np.int64).tobytes()).hexdigest())
    pred=None
    if record_predictions:
        pred=pd.DataFrame({'task_key':row['task_key'],'backbone':BACKBONE,
                           'method':row['method'],'sample_key':m.loc[test,'sample_key'].astype(str).to_numpy(),
                           'true_label':yt,'predicted_label':classes[np.argmax(prob,axis=1)],
                           'probability_vector':[json.dumps(z.tolist(),separators=(',',':')) for z in prob]})
    return row,audit,pred


## Phase C — Single-factor validation-based C selection

Run the three cells below. This full-run version already sets `C_CHECK_SMOKE_ONLY=False` in **Cell A** after the one-task check to evaluate all 60 tasks. Do not modify `C_GRID` or tolerances after observing test metrics.

**Warning:** a freshly opened Kaggle session requires Oxford feature cache or re-extraction; if the previous session remains open, paste Cells A–C directly into the previous NB15 to reuse `BASE_X`, `TARGETS`, and code definitions without another extraction.

In [ ]:
# Cell A — Run configuration and validation-only selectors
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import json, hashlib, time, warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# Step 1: True = 1 smoke task; Step 2: False = complete 60-task check.
C_CHECK_SMOKE_ONLY = False
C_GRID = (0.01, 0.1, 1.0, 10.0, 100.0)  # fixed before test comparisons
DELTA_F = 0.002
DELTA_N = 0.01
DELTA_E = 0.01

C_OUTPUT = ROOT / 'CAPFG15b_C_Validation_Tuning'
C_OUTPUT.mkdir(parents=True, exist_ok=True)
REQUIRED_CONTEXT = ['TARGETS','BASE_X','BASE_META','build_task_split',
    'support_fitted_pca','convert_with_pca','estimate_oxford_source_stats',
    'gen_dc_features','get_metrics','derive_seed']
missing=[k for k in REQUIRED_CONTEXT if k not in globals()]
if missing:
    raise RuntimeError('Run original NB15 preparation Cells 2-8 first. Missing: '+', '.join(missing))
if tuple(SHOTS)!=(5,10,20) or len(SEEDS)!=10 or len(TARGETS)!=2:
    raise RuntimeError('Target task specification differs from NB15.')
if len(BASE_X)!=8189 or BASE_X.shape[1]!=2048 or BASE_META.source_class_id.nunique()!=102:
    raise RuntimeError('Oxford auxiliary dataset differs from confirmed NB15 source.')
if tuple(C_GRID)!=(0.01,0.1,1.0,10.0,100.0):
    raise RuntimeError('Do not change C grid after seeing test results.')


def c_select_on_validation(metrics_df, kind):
    d = metrics_df.copy().reset_index(drop=True)
    assert len(d)==len(C_GRID) and set(d['C'])==set(C_GRID)
    eps=1e-12
    best_f=float(d.val_macro_f1.max())
    if kind=='F1-only':
        eligible=d.loc[d.val_macro_f1 >= best_f-eps]
        return eligible.sort_values('C',kind='stable').iloc[0]
    if kind=='F1-NLL-ECE':
        eligible=d.loc[d.val_macro_f1 >= best_f-DELTA_F-eps]
        min_n=float(eligible.val_nll.min())
        eligible=eligible.loc[eligible.val_nll <= min_n+DELTA_N+eps]
        min_e=float(eligible.val_ece.min())
        eligible=eligible.loc[eligible.val_ece <= min_e+DELTA_E+eps]
        return eligible.sort_values('C',kind='stable').iloc[0]
    raise ValueError(kind)

# Unit test: one selector retains best F1; one can trade small F1 for NLL/ECE.
_test = pd.DataFrame({
    'C':[0.01,0.1,1.,10.,100.],
    'val_macro_f1':[.590,.598,.600,.602,.597],
    'val_nll':[1.40,1.20,1.05,1.25,1.00],
    'val_ece':[.15,.12,.09,.10,.06]
})
assert float(c_select_on_validation(_test,'F1-only').C)==10.0
assert float(c_select_on_validation(_test,'F1-NLL-ECE').C)==1.0
print('Validation-only selector self-test: PASS')
print('Mode:', 'SMOKE one task' if C_CHECK_SMOKE_ONLY else 'FULL 60 tasks',
      '| C grid:',C_GRID,'| output:',C_OUTPUT)

In [ ]:
# Cell B — Generate each task's DC features ONCE; fit all C; select on validation; test selected only

def evaluate_dc_C_task(bundle,k,seed):
    t0=time.perf_counter()
    support,valid,test=build_task_split(bundle,k,seed)
    m=bundle.metadata
    classes=bundle.class_labels
    ys=m.loc[support,'label_canonical'].to_numpy(dtype=int)
    yv=m.loc[valid,'label_canonical'].to_numpy(dtype=int)
    yt=m.loc[test,'label_canonical'].to_numpy(dtype=int)
    assert not (set(support)&set(valid) or set(support)&set(test) or set(valid)&set(test))
    assert len(np.unique(ys))==len(classes)

    pca=support_fitted_pca(np.asarray(bundle.features[support],dtype=np.float64),seed)
    Zs=convert_with_pca(np.asarray(bundle.features[support],dtype=np.float64),pca)
    Zv=convert_with_pca(np.asarray(bundle.features[valid],dtype=np.float64),pca)
    # Do NOT transform or access test before validation selection.
    Zb=convert_with_pca(BASE_X,pca)
    _,base_mu,base_cov=estimate_oxford_source_stats(
        Zb,BASE_META.source_class_id.to_numpy(dtype=int))
    Xgen,ygen,source_neighbors=gen_dc_features(Zs,ys,base_mu,base_cov,seed)
    Xtrain=np.vstack([Zs,Xgen]); ytrain=np.concatenate([ys,ygen])
    assert len(Xgen)==len(ys)*SYNTH_PER_SUPPORT

    tk=f'{bundle.dataset}__k{k}__seed{seed}'
    cand=[]; trained={}
    for C in C_GRID:
        clf=LogisticRegression(
            C=float(C),solver='lbfgs',max_iter=LR_MAX_ITER,
            random_state=derive_seed(seed,'external_dc_fixed_lr'))
        clf.fit(Xtrain,ytrain)
        assert np.array_equal(clf.classes_,classes)
        valprob=clf.predict_proba(Zv)
        valmet=get_metrics(yv,valprob,classes)
        assert np.isfinite(valprob).all() and np.allclose(valprob.sum(axis=1),1,atol=1e-8)
        cand.append(dict(task_key=tk,backbone=BACKBONE,dataset=bundle.dataset,
                         shot=k,seed=seed,C=float(C),
                         **{f'val_{name}':value for name,value in valmet.items()}))
        trained[float(C)]=clf
    cand_df=pd.DataFrame(cand)
    selected=[]
    # Both selectors use the same fitted candidates and the same validation features.
    choices={name:c_select_on_validation(cand_df,name) for name in ('F1-only','F1-NLL-ECE')}
    # Test feature transform is deferred until after both C selections.
    Zt=convert_with_pca(np.asarray(bundle.features[test],dtype=np.float64),pca)
    for name,chosen in choices.items():
        C=float(chosen.C)
        prob=trained[C].predict_proba(Zt)
        testmet=get_metrics(yt,prob,classes)
        selected.append(dict(task_key=tk,backbone=BACKBONE,dataset=bundle.dataset,
                             shot=k,seed=seed,method=f'DC-Adapted-Oxford102-C-{name}',
                             selector=name,chosen_C=C,
                             **{f'val_{key}':float(chosen[f'val_{key}'])
                                for key in ('macro_f1','nll','ece','brier')},
                             **{f'test_{key}':value for key,value in testmet.items()},
                             n_support=len(support),n_validation=len(valid),
                             n_test=len(test),n_source_samples=len(BASE_X),
                             n_source_classes=len(base_mu),n_synthetic=len(Xgen),
                             pca_components=int(pca.n_components_),
                             elapsed_sec=round(time.perf_counter()-t0,3)))

    audit=dict(task_key=tk,backbone=BACKBONE,dataset=bundle.dataset,shot=k,seed=seed,
               n_support=len(support),n_val=len(valid),n_test=len(test),
               PCA_dimension=int(pca.n_components_),n_source_samples=len(BASE_X),
               n_synthetic=len(Xgen),C_candidates=len(C_GRID),
               test_labels_used_for_selection=False,validation_used_for_selection=True,
               target_other_train_used_for_base=False,
               source_target_image_hash_overlap_checked=False,
               support_sample_indices_sha256=hashlib.sha256(
                   np.asarray(support,dtype=np.int64).tobytes()).hexdigest(),
               test_sample_indices_sha256=hashlib.sha256(
                   np.asarray(test,dtype=np.int64).tobytes()).hexdigest(),
               same_synthetic_features_for_all_C=True,
               c_F1=float(choices['F1-only'].C),
               c_MO=float(choices['F1-NLL-ECE'].C))
    return cand,selected,audit

spec = [(ds,k,sd) for ds in TARGET_DATASETS for k in SHOTS for sd in SEEDS]
if C_CHECK_SMOKE_ONLY:
    spec=[SMOKE]

all_candidates=[];all_selected=[];all_audits=[]
for index,(ds,k,sd) in enumerate(spec,1):
    candidate,selected,audit=evaluate_dc_C_task(TARGETS[ds],k,sd)
    all_candidates.extend(candidate);all_selected.extend(selected);all_audits.append(audit)
    pd.DataFrame(all_candidates).to_csv(C_OUTPUT/'dc_C_candidate_VALIDATION.csv',index=False)
    pd.DataFrame(all_selected).to_csv(C_OUTPUT/'dc_C_selected_TEST.csv',index=False)
    pd.DataFrame(all_audits).to_csv(C_OUTPUT/'dc_C_protocol_audit.csv',index=False)
    print(f'[{index}/{len(spec)}] {audit["task_key"]} '
          f'C(F1)={audit["c_F1"]:g} C(MO)={audit["c_MO"]:g} '
          f'F1(MO)={selected[-1]["test_macro_f1"]:.4f}')
expected_candidates = len(C_GRID) * len(spec)
expected_selected = 2 * len(spec)

assert len(all_candidates) == expected_candidates
assert len(all_selected) == expected_selected
assert len({a['task_key'] for a in all_audits}) == len(spec)

if not C_CHECK_SMOKE_ONLY:
    assert len(spec) == 60
    assert len(all_candidates) == 60 * len(C_GRID)   # 300 for 5 C values
    assert len(all_selected) == 120                  # 2 selectors × 60 tasks

print('[COMPLETE]', len(spec), 'matched tasks',
      len(all_candidates), 'validation candidates',
      len(all_selected), 'selected test outcomes')

In [ ]:
# Cell C — Independent protocol audit, descriptive results, paired comparisons and ZIP
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import pandas as pd
import numpy as np
import json

cand_df=pd.DataFrame(all_candidates)
sel_df=pd.DataFrame(all_selected)
aud_df=pd.DataFrame(all_audits)
assert cand_df[['task_key','C']].duplicated().sum()==0
assert sel_df[['task_key','selector']].duplicated().sum()==0
assert set(sel_df['chosen_C'])<=set(C_GRID)
assert aud_df['PCA_dimension'].eq(64).all()
assert aud_df['same_synthetic_features_for_all_C'].all()
assert not aud_df['test_labels_used_for_selection'].any()

# Verify exact task sampling against the original DC evaluation, if that audit is attached.
old_audit_path=Path('/kaggle/working/CAPFG15_DC_Adapted_Output/dc_protocol_audit.csv')
if not old_audit_path.is_file():
    old_audit_path=find_input_file('dc_protocol_audit.csv',required=False)
if old_audit_path is not None and Path(old_audit_path).is_file():
    old_aud=pd.read_csv(old_audit_path)
    chk=aud_df.merge(old_aud[['task_key','support_sample_indices_sha256',
                              'test_sample_indices_sha256']],
                     on='task_key',how='left',validate='one_to_one',
                     suffixes=('_new','_original'))
    assert chk['support_sample_indices_sha256_original'].notna().all(),         'Missing original DC task audit rows'
    for kind in ('support','test'):
        assert (chk[f'{kind}_sample_indices_sha256_new'] ==
                chk[f'{kind}_sample_indices_sha256_original']).all(),             f'Original DC task {kind} indices differ: STOP.'
    print('MATCHED TASK AUDIT: PASSED',len(chk),'support/test index hashes')
else:
    print('Original DC protocol audit not attached: task-index hash verification deferred.')

config={
    'analysis':'Reviewer Major Issue 1 - DC C sensitivity',
    'experiment':'DC-Adapted-Oxford102',
    'backbone':BACKBONE,'run_full':not C_CHECK_SMOKE_ONLY,
    'task_count':len(spec),'C_grid':list(C_GRID),
    'selectors':{'F1-only':'validation Macro-F1 max; lower C tie-break',
                 'F1-NLL-ECE':'validation F1 tolerance 0.002; NLL 0.01; ECE 0.01; lower C tie-break'},
    'external_source':'Oxford Flowers102, labeled independent source classes',
    'test_access':'selected candidates only; no test labels used during selection',
    'caution':'adapted cross-domain DC; not original ICLR benchmark; no image hash overlap audit',
    'fixed_attributes':{'PCA_dim':PCA_D,'nearest_base_classes':BASE_NEIGHBORS,
                        'synthetic_per_support':SYNTH_PER_SUPPORT,
                        'cov_relative_ridge':COV_REL_RIDGE,'LR_max_iter':LR_MAX_ITER}}
(C_OUTPUT/'run_protocol.json').write_text(json.dumps(config,indent=2),encoding='utf-8')

summary=(sel_df.groupby(['selector','dataset','shot'],as_index=False)
         [['test_macro_f1','test_nll','test_ece','test_brier']]
         .agg(['mean','std']))
summary.columns=['_'.join([str(v) for v in t if v!='']).strip('_')
                 if isinstance(t,tuple) else t for t in summary.columns]
summary.to_csv(C_OUTPUT/'dc_C_summary_by_dataset_shot.csv',index=False)
(sel_df.groupby(['selector','chosen_C']).size().rename('task_count').reset_index()
 .to_csv(C_OUTPUT/'dc_C_choice_frequencies.csv',index=False))

# Optional pair CA-PFG and original fixed C=1 from the previously saved evaluation.
reference_path = (Path(MANUAL['test_reference']) if MANUAL.get('test_reference')
                  else find_input_file('normalized_all_results_TEST_ONLY.csv',required=False))
if reference_path is not None:
    ref=pd.read_csv(reference_path)
    ref=ref.loc[(ref['backbone']=='ResNet-50')&(ref['method']=='CA-PFG-MO')]
    assert len(ref)==60 and not ref.duplicated(['dataset','shot','seed']).any()
    pairs=sel_df.merge(ref[['dataset','shot','seed','test_macro_f1','test_nll','test_ece','test_brier']],
        on=['dataset','shot','seed'],how='left',suffixes=('_dcC','_capfg'),validate='many_to_one')
    assert pairs['test_macro_f1_capfg'].notna().all()
    for metric in ('macro_f1','nll','ece','brier'):
        ca=pairs[f'test_{metric}_capfg'];dc=pairs[f'test_{metric}_dcC']
        pairs[f'CA_PFG_favorable_delta_{metric}']=(ca-dc) if metric=='macro_f1' else (dc-ca)
    pairs.to_csv(C_OUTPUT/'paired_DC_C_vs_CA_PFG_TEST.csv',index=False)
    print('Paired comparison to CA-PFG available:',len(pairs),'method-task comparisons')
else:
    print('CA-PFG corrected test reference not attached; main C-sensitivity outputs remain valid.')

fixed_path=Path('/kaggle/working/CAPFG15_DC_Adapted_Output/dc_adapted_task_results.csv')
if not fixed_path.is_file():
    fixed_path=find_input_file('dc_adapted_task_results.csv',required=False)
if fixed_path is not None and Path(fixed_path).is_file():
    fixed=pd.read_csv(fixed_path)
    fixed=fixed[['dataset','shot','seed','test_macro_f1','test_nll','test_ece','test_brier']].copy()
    assert len(fixed)==60 and not fixed.duplicated(['dataset','shot','seed']).any()
    ck=sel_df.merge(fixed,on=['dataset','shot','seed'],how='left',
                    suffixes=('_tuned','_fixedC1'),validate='many_to_one')
    for metric in ('macro_f1','nll','ece','brier'):
        a=ck[f'test_{metric}_tuned'];b=ck[f'test_{metric}_fixedC1']
        ck[f'tuned_favorable_delta_{metric}']=(a-b) if metric=='macro_f1' else (b-a)
    ck.to_csv(C_OUTPUT/'paired_DC_Tuned_vs_FixedC1_TEST.csv',index=False)
    print('Paired check vs original C=1 available:',len(ck),'method-task comparisons')

(C_OUTPUT/'README.txt').write_text(
    'DC-Adapted-Oxford102 C validation-only check. '
    'No new target data, no CA-PFG refit. '
    'The original fixed-C DC and CA-PFG test results are external reference data only.\n',
    encoding='utf-8')
zip_path=ROOT/'CAPFG15b_DC_C_Validation_Tuning_Results.zip'
with ZipFile(zip_path,'w',ZIP_DEFLATED) as archive:
    for f in sorted(C_OUTPUT.iterdir()):
        if f.is_file(): archive.write(f,arcname=f.name)
print('ZIP:',zip_path, '| size',round(zip_path.stat().st_size/1024,2),'KB')
print('Deliver this ZIP and note whether RUN was SMOKE or FULL.')